In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Input data is in MWh. It is divided by $10^3$ to convert it to GWh.

In [ ]:
demand_base = pd.read_csv('norway_demand_2006-2015.csv', parse_dates=['datetime'], index_col='datetime')/1000

# Scale demand

In [ ]:
average_demand = demand_base.sum().sum()/(demand_base.index.year.to_series().unique().size)
average_demand/1000

In [ ]:
demand_scaled = demand_base*(140000/average_demand) # scale the demand so that the average equals the demand of 2022 (140 TWh)

In [ ]:
demand = demand_scaled.copy()

In [ ]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

# Regular consumption and losses
Subtract 5 TWh (-3.57% of each hour in each county)

In [ ]:
demand = demand - (5/140)*demand

In [ ]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

# Electric transport
Add 19 TWh

In [ ]:
# OLD CODE
# demand = demand + np.ones(11)*((23*10**3)/(365*24*11))
# base_load = ([4.088515615, 4.880937396, 2.40344075, 1.829410414, 18.6444199, 6.106533718, 7.141278395, 4.349801459, 6.770179305, 4.947604974, 1.851576703])
# load_profile = np.array([[0.049621531], [0.033761865], [0.022347711], [0.014898474], [0.009852217], [0.006848492], [0.004805959], [0.004445512], [0.007929833], [0.008410429], [0.008410429], [0.011294005], [0.015018623], [0.018863391], [0.023068605], [0.046737955], [0.092154271], [0.102607233], [0.085906524], [0.0887901], [0.094076655], [0.091793824], [0.085786375], [0.072569987]])
# load_profile_fylke = load_profile*base_load
# for i in range(0,24):
#     demand.loc[demand.index.hour == i] = demand[demand.index.hour == i] + load_profile_fylke[i]

In [ ]:
dist = pd.read_csv('2024-04-12 15-11-52 - eksport fra SINTEF energikart, fylker (2060 kollektiv og gods, strømforbruk Wt per døgn).csv', sep=';')
dist = dist.set_index('NO' + dist['county number']).drop(columns=['total number of cells', 'number of sources', 'county number', 'county name'], index=['NOUnknown'])

In [ ]:
transport_demand = ((19*(10**3))/(365*24))*(dist/dist.sum())

In [ ]:
demand = demand.add(transport_demand.T.values)

In [ ]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

# Industry
Add 50 TWh

In [ ]:
# from https://www.ssb.no/statbank/table/10314/tableViewLayout1/ (data from 2019, later years has no data for Nordland)
electricity_industry = pd.Series({'NO03':888.8, 'NO11':6424.1, 'NO15':8867.4, 'NO18':6348.9, 'NO30':4142.4, 'NO34':1091.6, 'NO38':3975.6, 'NO42':4581.8, 'NO46':12720.9, 'NO50':3946.9, 'NO54':3285.1})

In [ ]:
industry = (((electricity_industry/electricity_industry.sum())*50*10**3)/(365*24)).to_numpy()

In [ ]:
#industry = [0.074396516, 0.960092477, 0.819807236, 0.958079203, 0.47070081, 0.109119971, 1.142689061, 0.471555006, 1.893724859, 0.526302915, 0.564399524]

In [ ]:
demand = demand + industry

In [ ]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

# Petroleum
Add 11 TWh

In [ ]:
electricity_petroleum = pd.Series({'NO03':0, 'NO11':3700, 'NO15':2050, 'NO18':0, 'NO30':0, 'NO34':0, 'NO38':0, 'NO42':200, 'NO46':4900, 'NO50':1600, 'NO54':4400})

In [ ]:
petroleum = ((electricity_petroleum/electricity_petroleum.sum())*11*1000)/(365*24)

In [ ]:
demand = demand + petroleum

In [ ]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

# Battery production and data centres
Add 26 TWh

In [ ]:
electricity_battery_GWh = pd.Series({'NO03':0, 'NO11':40, 'NO15':0, 'NO18':29, 'NO30':0, 'NO34':0, 'NO38':0, 'NO42':43, 'NO46':0, 'NO50':40, 'NO54':0})*65
electricity_data_GWh = pd.Series({'NO03':0, 'NO11':240, 'NO15':0, 'NO18':0, 'NO30':0, 'NO34':150, 'NO38':840, 'NO42':0, 'NO46':0, 'NO50':0, 'NO54':0})*0.8*365*24/1000

In [ ]:
data_battery = electricity_battery_GWh + electricity_data_GWh

Distribute the remaining demand (26 TWh minus the total known demand from battery factories and data centres above) equally between all counties.

In [ ]:
batt_n_data_yearly = data_battery + (26*1000 - data_battery.sum())/11

In [ ]:
batt_n_data = batt_n_data_yearly/(365*24)

In [ ]:
demand = demand + batt_n_data

In [ ]:
demand.sum().sum()/(demand_base.index.year.to_series().unique().size*1000) # calculate average

# To CSV

In [ ]:
demand = demand*1000 # convert back to MWh

In [ ]:
(demand.sum().sum())/(demand_base.index.year.to_series().unique().size) # calculate average (MWh)

In [ ]:
demand = demand.round(1)

In [ ]:
demand

In [ ]:
demand.to_csv('demand_2040(MWh).csv')

# Directly to .dd file

For 2010

In [ ]:
demand2010 = demand.loc['2010']

In [ ]:
demand2010 = demand2010.reset_index().drop(columns=['datetime']).melt(ignore_index=False)

In [ ]:
demand2010 = demand2010.set_index(demand2010.variable + '.' + demand2010.index.astype(str)).drop(columns=['variable']).reset_index() #.rename_axis('variable')

In [ ]:
demand2010['value'] = demand2010['value']*10**3

In [ ]:
demand2010

In [ ]:
demand2010.to_csv('BASE_demand_2010.dd', sep=' ', lineterminator='\n')

In [ ]:
    np.savetxt("BASE_demand_2010_2.dd", demand2010, delimiter=" ", fmt="%s", header="parameter \ndemand / ", footer="/ \n ", comments="")

In [ ]:
demand2010.sum()/10**6

# Read .dd file

In [ ]:
read = pd.read_csv('BASE_demand_2010.dd', sep=' ', skiprows=[0,1,96362,96363], header=None)

In [ ]:
read.mean()

In [ ]:
demand2010.mean()